# Cybersecurity — Side-channel Periodicity (Toy)

Simulate a timing/power trace with round-dependent leakage that repeats every *r* operations.
We estimate *r* and visualize with QFT peaks (explanatory tool). No real crypto broken here; synthetic data only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft, rfftfreq
from quantum_hybrid_system import PeriodicState

rng = np.random.default_rng(4)
T = 5000
r_true = 10
trace = 0.3*np.sin(2*np.pi*np.arange(T)/r_true) + rng.normal(0, 1.0, size=T)

yf = np.abs(rfft(trace - trace.mean()))
xf = rfftfreq(T, d=1.0)
k = np.argmax(yf[1:]) + 1
freq = xf[k]
period_est = 1/freq if freq>0 else np.nan
print("Estimated leakage period ~", period_est)

plt.figure()
plt.plot(trace[:400])
plt.title("Synthetic side-channel trace (first 400 samples)")
plt.xlabel("t")
plt.ylabel("signal")
plt.show()

n = 10
r = max(2, min(int(round(period_est)), 64))
ps = PeriodicState(num_qubits=n, period=r)
samples = ps.measure(num_shots=4000, use_qft=True)

bins = 64
hist = np.zeros(bins, dtype=int)
N = 2**n
for s in samples:
    hist[(s * bins) // N] += 1

plt.figure()
plt.bar(np.arange(bins), hist)
plt.title(f"QFT histogram (periodicity r≈{r})")
plt.xlabel("coarse frequency bin")
plt.ylabel("counts")
plt.show()